Import Python libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import sklearn as sk

print("pandas: ", pd.__version__)
print("numpy: ", np.__version__)
print("matplotlib: ", mpl.__version__)
print("sklearn: ", sk.__version__)

Load dataset from archive


In [ ]:
ds = pd.read_csv("datasets/archive/games.csv")
ds.shape
ds.info()
print("Null values: " % (ds.isnull().sum()))

In [ ]:
fake_news = pd.read_csv("datasets/FakeNewsNet.csv")
#what are the columns and types?
fake_news.info()

#what does the data look like?
fake_news.head(10)
fake_news.sample(10, random_state=42)

#how much data  is missing?
print(f"Missing data: {fake_news.isna().mean().sort_values(ascending=False)}")

print(f"Missing values per column: {fake_news.isna().sum().sum()}")

for col in fake_news.columns:
    if fake_news[col].nunique() < 20:
        print(f"\n{col}:")
        print(fake_news[col].value_counts().head(10))

In [ ]:
# 1. Is it unique?
print("Unique values:", fake_news["tweet_num"].nunique())
print("Total rows:", len(fake_news))

# 2. What's the range?
print(fake_news["tweet_num"].describe())

# 3. Is it sequential?
print(fake_news["tweet_num"].head(20))

# 4. Does it correlate with row order?
print(fake_news["tweet_num"].is_monotonic_increasing)

New dataset: news_sample.csv

In [ ]:
import glob
from collections import Counter
import pandas as pd

news = pd.read_csv("datasets/news_cleaned_2018_02_13.csv", nrows=0)
print(news.columns.tolist())

import json

f = "datasets/news_cleaned_2018_02_13.csv"
total_counts = Counter()

for chunk in pd.read_csv(
    f,
    chunksize=100_000,
    usecols=["type"],
    engine="python",
    on_bad_lines="skip",
    nrows=500_000
):
    total_counts.update(chunk["type"].value_counts().to_dict())
# Save to file
with open("datasets/full_type_distribution.json", "w") as out:
    json.dump(dict(total_counts), out, indent=2)

# Print
print("\nDataset type distribution:")
for label, count in total_counts.most_common():
    print(f"{label:15s} {count:>10,}")

#Check: how many distinct types does each domain have?
domains_per_type = news.groupby("domain")["type"].nunique()

# If every domain has exactly 1 type, this will all be 1s
print("Max types per domain:", domains_per_type.max())
print("Min types per domain:", domains_per_type.min())
print("Number of domains with more than 1 type:", (domains_per_type > 1).sum())

# Alternative view: for each domain, what types appear?
print("\nSample of domain → type mappings:")
print(news.groupby("domain")["type"].unique().head(20))


In [ ]:
import csv
import sys
import pandas as pd

csv.field_size_limit(sys.maxsize)

f = "datasets/news_cleaned_2018_02_13.csv"

chunks = []
for chunk in pd.read_csv(
    f,
    chunksize=100_000,
    usecols=["domain", "type"],
    engine="python",
    on_bad_lines="skip",
    nrows=500_000
):
    chunks.append(chunk)
   

sample = pd.concat(chunks, ignore_index=True)
print("Sample shape:", sample.shape)

domains_per_type = sample.groupby("domain")["type"].nunique()

print("\nMax types per domain:", domains_per_type.max())
print("Min types per domain:", domains_per_type.min())
print("Domains with more than 1 type:", (domains_per_type > 1).sum())
print("Total unique domains:", sample["domain"].nunique())

print("\nSample domain → type mappings:")
print(sample.groupby("domain")["type"].unique().head(20))